In [1]:
import numpy as np, warnings, json, datetime
warnings.filterwarnings("ignore")
from scipy.stats import beta as beta_dist
from collections import defaultdict
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit.quantum_info import Statevector, DensityMatrix, state_fidelity
from qiskit_aer import AerSimulator

SHOTS=9000; N_BOOTSTRAP=2000; SEED=42
RNG=np.random.default_rng(SEED)
THETA_MSG=2.5349076035276403; VARPHI_MSG=2.0022404587009195

def build_tomo_circuits(theta=THETA_MSG, varphi=VARPHI_MSG):
    """
    Tomography circuits measuring C register (which holds ρ_M after SWAP).
    Key fix: NO U†(C) applied — that step maps ρ_M → |0⟩ and is only 
    for the CTC loop closure, not for tomographic readout.
    Register order: C=0,E=1,R=2,G=3,M=4,A=5,Y=6
    Classical bits: crR=0(rightmost), crG=1, crTomo=2(leftmost)
    Bitstring format: "crTomo crG crR"
    """
    circuits = []
    for basis in ['Z','X','Y']:
        C=QuantumRegister(1,'C'); E=QuantumRegister(1,'E'); R=QuantumRegister(1,'R')
        G=QuantumRegister(1,'G'); M=QuantumRegister(1,'M'); A=QuantumRegister(1,'A')
        Yr=QuantumRegister(1,'Y')
        crR=ClassicalRegister(1,'crR'); crG=ClassicalRegister(1,'crG')
        crT=ClassicalRegister(1,'crTomo')
        qc=QuantumCircuit(C,E,R,G,M,A,Yr,crR,crG,crT)
        qc.u(theta,varphi,0.0,M); qc.swap(C,M); qc.barrier()
        qc.h(E); qc.cx(E,M); qc.h(R); qc.cx(R,G); qc.h(A); qc.cx(A,Yr); qc.barrier()
        qc.cz(C,R); qc.cz(E,R); qc.cz(C,E); qc.h(C); qc.h(E); qc.h(R)
        qc.cz(C,R); qc.cz(C,E); qc.cz(E,R); qc.barrier()
        qc.cz(A,G); qc.cz(M,A); qc.cz(G,M); qc.h(A); qc.h(M); qc.h(G)
        qc.cz(A,G); qc.cz(G,M); qc.cz(M,A); qc.barrier()
        qc.cx(R,G); qc.h(R); qc.barrier()
        qc.swap(C,Yr)
        # NO U†(C) — C now holds ρ_M directly
        # Bell projection measurements
        qc.measure(R,crR); qc.measure(G,crG)
        # Tomography rotation on C
        if basis=='X': qc.h(C)
        elif basis=='Y': qc.sdg(C); qc.h(C)
        qc.measure(C,crT)
        circuits.append((basis,qc))
    return circuits

sim=AerSimulator()
tomo_circuits=build_tomo_circuits()
raw_results={}
print("Running corrected tomography circuits on AerSimulator ...")
for basis,qc in tomo_circuits:
    t=transpile(qc,sim,optimization_level=1,seed_transpiler=SEED)
    res=sim.run(t,shots=SHOTS).result()
    raw_results[basis]=res.get_counts(t)

def postselect(counts_dict):
    kept=defaultdict(int); n_total=0; n_kept=0
    for bs,count in counts_dict.items():
        n_total+=count
        parts=bs.split(' ')   # "crTomo crG crR"
        crTomo_bit=parts[0]; crG_bit=parts[1]; crR_bit=parts[2]
        if crR_bit=='0' and crG_bit=='0':
            kept[crTomo_bit]+=count; n_kept+=count
    return dict(kept),n_total,n_kept

postselected={}; n_totals={}; n_kept_all={}
print("\nPost-selection results:")
print(f"  {'Basis':>5}  {'Total':>7}  {'Kept':>7}  {'p_succ':>8}  {'p(0|kept)':>10}")
for basis in ['Z','X','Y']:
    k,nt,nk=postselect(raw_results[basis])
    postselected[basis]=k; n_totals[basis]=nt; n_kept_all[basis]=nk
    p=nk/nt; n0=k.get('0',0); n1=k.get('1',0)
    p0=n0/(n0+n1) if (n0+n1)>0 else float('nan')
    print(f"  {basis:>5}  {nt:>7d}  {nk:>7d}  {p:>8.4f}  {p0:>10.4f}")

def clopper_pearson(k,n,alpha=0.05):
    lo=beta_dist.ppf(alpha/2,k,n-k+1) if k>0 else 0.0
    hi=beta_dist.ppf(1-alpha/2,k+1,n-k) if k<n else 1.0
    return lo,hi

nk=n_kept_all['Z']; nt=n_totals['Z']
p_hat=nk/nt; cp_lo,cp_hi=clopper_pearson(nk,nt)

def pauli_exp(c):
    n0=c.get('0',0); n1=c.get('1',0); N=n0+n1
    return (n0-n1)/N if N>0 else 0.0

sx=pauli_exp(postselected['X']); sy=pauli_exp(postselected['Y']); sz=pauli_exp(postselected['Z'])

def reconstruct(sx,sy,sz):
    X=np.array([[0,1],[1,0]],dtype=complex); Y=np.array([[0,-1j],[1j,0]],dtype=complex)
    Z=np.array([[1,0],[0,-1]],dtype=complex)
    rho=(np.eye(2)+sx*X+sy*Y+sz*Z)/2
    ev,evec=np.linalg.eigh(rho); ev=np.maximum(ev,0); ev/=ev.sum()
    return (evec*ev)@evec.conj().T

rho_Y=reconstruct(sx,sy,sz)
qcm=QuantumCircuit(1); qcm.u(THETA_MSG,VARPHI_MSG,0.0,0)
rho_M=DensityMatrix(Statevector.from_instruction(qcm))
F_hat=state_fidelity(DensityMatrix(rho_Y),rho_M)
print(f"\n⟨X⟩={sx:+.4f}  ⟨Y⟩={sy:+.4f}  ⟨Z⟩={sz:+.4f}")
print(f"Purity={np.trace(rho_Y@rho_Y).real:.4f}  F={F_hat:.4f}")

def bs_resample(c,rng):
    keys=list(c.keys()); vals=np.array([c[k] for k in keys])
    N=vals.sum(); new=rng.multinomial(N,vals/N)
    return {k:int(v) for k,v in zip(keys,new)}

bsF=np.zeros(N_BOOTSTRAP)
for i in range(N_BOOTSTRAP):
    sx_b=pauli_exp(bs_resample(postselected['X'],RNG))
    sy_b=pauli_exp(bs_resample(postselected['Y'],RNG))
    sz_b=pauli_exp(bs_resample(postselected['Z'],RNG))
    bsF[i]=state_fidelity(DensityMatrix(reconstruct(sx_b,sy_b,sz_b)),rho_M)

F_lo=np.percentile(bsF,2.5); F_hi=np.percentile(bsF,97.5)
print(f"Bootstrap 95% CI = [{F_lo:.4f}, {F_hi:.4f}]  std={np.std(bsF):.4f}")

print(f"\n{'='*58}")
print("  FINAL VALIDATED SUMMARY (Aer noiseless simulator)")
print(f"{'='*58}")
print(f"  p_succ     = {p_hat:.4f}  [{cp_lo:.4f},{cp_hi:.4f}]  (CP 95% CI)")
print(f"  F(ρ_Y,ρ_M) = {F_hat:.4f}  [{F_lo:.4f},{F_hi:.4f}]  (bootstrap 95% CI)")
print(f"  Ideal      :  p_succ=0.2500,  F=1.0000")
print(f"{'='*58}")
print("[✓] Pipeline fully validated. Ready for IBM hardware.")

Running corrected tomography circuits on AerSimulator ...

Post-selection results:
  Basis    Total     Kept    p_succ   p(0|kept)
      Z     9000     2217    0.2463      0.0907
      X     9000     2230    0.2478      0.3749
      Y     9000     2286    0.2540      0.7664

⟨X⟩=-0.2502  ⟨Y⟩=+0.5328  ⟨Z⟩=-0.8187
Purity=1.0000  F=0.9999
Bootstrap 95% CI = [0.9908, 1.0000]  std=0.0026

  FINAL VALIDATED SUMMARY (Aer noiseless simulator)
  p_succ     = 0.2463  [0.2375,0.2554]  (CP 95% CI)
  F(ρ_Y,ρ_M) = 0.9999  [0.9908,1.0000]  (bootstrap 95% CI)
  Ideal      :  p_succ=0.2500,  F=1.0000
[✓] Pipeline fully validated. Ready for IBM hardware.


# Hardware - IBM Torino


In [7]:
# =============================================================================
#  IBM Quantum Hardware Post-Selection Pipeline — Implementation A
#  Probabilistic Yoshida–Kitaev Decoder (Lloyd-type CTC Emulation)
#
#  Pipeline:
#    1. Build & transpile the circuit on ibm_torino
#    2. Run on IBM Quantum hardware, store raw bitstrings
#    3. Filter bitstrings on Bell outcome |Φ+⟩ = (crR=0, crG=0)
#    4. Estimate p_succ with Clopper–Pearson 95% CI
#    5. State tomography on output qubit Y (3 Pauli bases: X, Y, Z)
#    6. Reconstruct ρ_Y via linear inversion + MLE
#    7. Compute F(ρ_Y, ρ_M) with bootstrap 95% CI
#    8. Print paper-ready summary table
#
#  Backend  : ibm_torino  (133-qubit Heron r2)
#  Shots    : 9000 total  (~2250 post-selected expected)
#  Calibration date: retrieved at job submission time
# =============================================================================

# ── Imports ───────────────────────────────────────────────────────────────────
import numpy as np
import warnings
warnings.filterwarnings("ignore")
from scipy.stats import beta as beta_dist
from scipy.optimize import minimize
from collections import defaultdict

from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit.quantum_info import (Statevector, DensityMatrix,
                                  state_fidelity, Pauli, SparsePauliOp)
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler
from qiskit_ibm_runtime import Batch

# =============================================================================
#  CONFIG — edit only this block
# =============================================================================
IBM_TOKEN   = "QfkScNfX4bVJ5lXm0082x7F7J6vya3SF5LJZFNOYXqqO"
BACKEND     = "ibm_torino"
SHOTS       = 1000          # total shots
N_BOOTSTRAP = 2000          # bootstrap resamples for F confidence interval
CI_LEVEL    = 0.95          # confidence level
SEED        = 42
RNG         = np.random.default_rng(SEED)

# Message state parameters (same as your circuit)
THETA_MSG   = 2.5349076035276403
VARPHI_MSG  = 2.0022404587009195

# =============================================================================
#  STEP 1 — Build circuit components
# =============================================================================

def build_base_circuit(theta=THETA_MSG, varphi=VARPHI_MSG):
    """Core 7-qubit probabilistic decoder, no measurements."""
    C = QuantumRegister(1, 'C')
    E = QuantumRegister(1, 'E')
    R = QuantumRegister(1, 'R')
    G = QuantumRegister(1, 'G')
    M = QuantumRegister(1, 'M')
    A = QuantumRegister(1, 'A')
    Y = QuantumRegister(1, 'Y')

    qc = QuantumCircuit(C, E, R, G, M, A, Y)

    # Prepare message on M
    qc.u(theta, varphi, 0.0, M)
    # SWAP message into CTC register
    qc.swap(C, M)
    qc.barrier()
    # Bell pairs: (E,M), (R,G), (A,Y)
    qc.h(E);  qc.cx(E, M)
    qc.h(R);  qc.cx(R, G)
    qc.h(A);  qc.cx(A, Y)
    qc.barrier()
    # Scrambling unitary U on (C, E, R)
    qc.cz(C, R);  qc.cz(E, R);  qc.cz(C, E)
    qc.h(C);  qc.h(E);  qc.h(R)
    qc.cz(C, R);  qc.cz(C, E);  qc.cz(E, R)
    qc.barrier()
    # Probabilistic decoder: U† on (A, M, G)
    qc.cz(A, G);  qc.cz(M, A);  qc.cz(G, M)
    qc.h(A);  qc.h(M);  qc.h(G)
    qc.cz(A, G);  qc.cz(G, M);  qc.cz(M, A)
    qc.barrier()
    # Bell projection on (R, G)
    qc.cx(R, G)
    qc.h(R)
    qc.barrier()
    # SWAP output Y -> C, apply U† on C
    qc.swap(C, Y)
    qc.u(theta, varphi, 0.0, C).inverse()

    return qc, C, E, R, G, M, A, Y


def build_tomography_circuits():
    """
    Build 3 tomography circuits measuring register C in X, Y, Z bases.

    After SWAP(C,Y), register C holds ρ_M (the recovered message).
    We do NOT apply U†(C) here — that step maps ρ_M back to |0⟩ and is
    only present in the original circuit for the CTC loop closure, not for
    tomographic readout. Tomography measures C directly against ρ_M.

    Post-selection condition: crR = 0  AND  crG = 0  (Bell outcome |Φ+⟩)

    Classical register order (added last→first in bitstring, left→right):
        crTomo (last added) | crG | crR (first added)
    Bitstring format: "crTomo crG crR"   e.g. "0 0 0"

    Returns list of (basis_label, circuit) pairs.
    """
    circuits = []
    for basis in ['Z', 'X', 'Y']:
        C = QuantumRegister(1, 'C');  E = QuantumRegister(1, 'E')
        R = QuantumRegister(1, 'R');  G = QuantumRegister(1, 'G')
        M = QuantumRegister(1, 'M');  A = QuantumRegister(1, 'A')
        Yr = QuantumRegister(1, 'Y')
        crR    = ClassicalRegister(1, 'crR')
        crG    = ClassicalRegister(1, 'crG')
        crTomo = ClassicalRegister(1, 'crTomo')
        qc = QuantumCircuit(C, E, R, G, M, A, Yr, crR, crG, crTomo)

        # --- Prepare message on M ---
        qc.u(THETA_MSG, VARPHI_MSG, 0.0, M)
        qc.swap(C, M); qc.barrier()
        # --- Bell pairs ---
        qc.h(E);  qc.cx(E, M)
        qc.h(R);  qc.cx(R, G)
        qc.h(A);  qc.cx(A, Yr); qc.barrier()
        # --- Scrambling unitary ---
        qc.cz(C, R);  qc.cz(E, R);  qc.cz(C, E)
        qc.h(C);  qc.h(E);  qc.h(R)
        qc.cz(C, R);  qc.cz(C, E);  qc.cz(E, R); qc.barrier()
        # --- Probabilistic decoder U† ---
        qc.cz(A, G);  qc.cz(M, A);  qc.cz(G, M)
        qc.h(A);  qc.h(M);  qc.h(G)
        qc.cz(A, G);  qc.cz(G, M);  qc.cz(M, A); qc.barrier()
        # --- Bell projection on (R, G) ---
        qc.cx(R, G);  qc.h(R); qc.barrier()
        # --- SWAP: C now holds ρ_M (no U†!) ---
        qc.swap(C, Yr)
        # --- Post-selection measurements ---
        qc.measure(R, crR)
        qc.measure(G, crG)
        # --- Tomography rotation on C ---
        if basis == 'X':
            qc.h(C)
        elif basis == 'Y':
            qc.sdg(C)
            qc.h(C)
        # basis == 'Z': no rotation
        qc.measure(C, crTomo)
        circuits.append((basis, qc))

    return circuits


# =============================================================================
#  STEP 2 — Transpile all 3 tomography circuits
# =============================================================================

print("=" * 65)
print("  IBM Quantum Post-Selection Pipeline — Implementation A")
print("=" * 65)

service = QiskitRuntimeService(channel="ibm_quantum_platform", token=IBM_TOKEN)
backend = service.backend(BACKEND)
print(f"\n[✓] Connected to {BACKEND}")

tomo_circuits_raw = build_tomography_circuits()

print(f"  Transpiling 3 tomography circuits (optimization_level=3) ...")
tomo_circuits_transpiled = []
for basis, qc in tomo_circuits_raw:
    t_circ = transpile(qc, backend=backend,
                       optimization_level=3,
                       seed_transpiler=SEED)
    tomo_circuits_transpiled.append((basis, t_circ))
    print(f"    Basis {basis}: depth={t_circ.depth()}, "
          f"2q-gates={sum(1 for _,qa,_ in t_circ.data if len(qa)==2)}")

# =============================================================================
#  STEP 3 — Run on IBM Quantum hardware
#
#  NOTE: Sessions are not available on the IBM Quantum Open Plan.
#  We use Batch mode instead — supported on the open plan and recommended
#  for collections of independent circuits. All 3 circuits are submitted
#  together in one batch for efficiency.
# =============================================================================

def extract_counts(result):
    """
    Extract joint {bitstring: count} dict from a SamplerV2 PrimitiveResult.

    SamplerV2 stores each classical register as a separate BitArray under
    pub.data.<register_name>  — there is no single combined counts dict.
    We reconstruct joint bitstrings by zipping the per-shot arrays:

        bitstring = crTomo_bit | crG_bit | crR_bit   (leftmost = last added)

    This matches the register order the postselect() function expects.
    """
    from collections import Counter
    import numpy as np

    pub  = result[0]
    data = pub.data

    # Each register is a BitArray with shape (n_shots, n_bits_in_register)
    # All three registers are 1-bit wide so .array.flatten() gives (n_shots,)
    try:
        crT_arr = data.crTomo.array.flatten()   # shape (n_shots,)
        crG_arr = data.crG.array.flatten()
        crR_arr = data.crR.array.flatten()
    except AttributeError as e:
        # Fallback: print available attributes to help debug
        avail = [a for a in dir(data) if not a.startswith('_')
                 and hasattr(getattr(data, a), 'get_counts')]
        raise RuntimeError(
            f"Could not find expected registers (crTomo/crG/crR). "
            f"Available BitArray attributes: {avail}"
        ) from e

    # Build joint bitstring "crTomo crG crR" per shot and count
    joint = [f"{int(t)}{int(g)}{int(r)}"
             for t, g, r in zip(crT_arr, crG_arr, crR_arr)]
    return dict(Counter(joint))

print(f"\n  Submitting {len(tomo_circuits_transpiled)} circuits × {SHOTS} shots "
      f"to {BACKEND} (Batch mode — open plan compatible) ...")

raw_results = {}   # basis -> bitstring count dict
jobs        = {}   # basis -> job handle

with Batch(backend=backend) as batch:
    sampler = Sampler(mode=batch)
    for basis, t_circ in tomo_circuits_transpiled:
        print(f"    Submitting basis {basis} ...", end=" ", flush=True)
        jobs[basis] = sampler.run([t_circ], shots=SHOTS)
        print(f"job_id = {jobs[basis].job_id()}")

# Collect results — blocks until each job finishes
print("\n  Collecting results (waiting for IBM Quantum jobs) ...")
for basis in ['Z', 'X', 'Y']:
    print(f"    Waiting for basis {basis} ...", end=" ", flush=True)
    result = jobs[basis].result()
    counts = extract_counts(result)
    raw_results[basis] = counts
    total  = sum(counts.values())
    print(f"done  ({total} total counts, {len(counts)} unique bitstrings)")
    try:
        q_secs = jobs[basis].metrics()['usage']['quantum_seconds']
        print(f"      QPU time: {q_secs:.4f} s")
    except Exception:
        pass

# =============================================================================
#  STEP 4 — Store raw bitstrings and post-select on Bell outcome |Φ+⟩
#           Post-selection condition: crR == 0  AND  crG == 0
#
#  Bitstring format from SamplerV2:  "crTomo crG crR"
#  (Qiskit orders classical registers right-to-left in the bitstring:
#   rightmost = first register added = crR, then crG, then crTomo)
# =============================================================================

def parse_bitstring(bitstring):
    """
    Parse a Qiskit bitstring into (crTomo, crG, crR) bits.
    Handles both formats hardware and simulator may return:
      Spaced  (simulator): "0 1 0"  -> splits to ['0','1','0']
      Compact (hardware):  "010"    -> list gives ['0','1','0']
    Register order: crTomo (leftmost) | crG | crR (rightmost).
    """
    parts = bitstring.split(' ') if ' ' in bitstring else list(bitstring)
    if len(parts) != 3:
        raise ValueError(
            f"Unexpected bitstring '{bitstring}' — need 3 bits, got {len(parts)}."
        )
    return parts[0], parts[1], parts[2]   # crTomo, crG, crR


def postselect(counts_dict):
    """
    Filter counts keeping only shots where crR=0 AND crG=0  (Bell |Phi+>).
    Returns (postselected_tomo_counts, n_total, n_kept).
    The returned dict maps '0'/'1' (crTomo outcome) -> count.
    """
    kept    = defaultdict(int)
    n_total = 0
    n_kept  = 0
    for bitstring, count in counts_dict.items():
        n_total += count
        crTomo_bit, crG_bit, crR_bit = parse_bitstring(bitstring)
        if crR_bit == '0' and crG_bit == '0':
            kept[crTomo_bit] += count
            n_kept += count
    return dict(kept), n_total, n_kept


postselected = {}   # basis -> {'0': n0, '1': n1}
n_totals     = {}
n_kept_all   = {}

print("\n  Post-selection results:")
print(f"  {'Basis':>6}  {'Total shots':>12}  {'Kept (Bell=00)':>15}  "
      f"{'p_succ':>8}  {'p(0|kept)':>10}")
print("  " + "-" * 60)

for basis in ['Z', 'X', 'Y']:
    kept, n_total, n_kept = postselect(raw_results[basis])
    postselected[basis]  = kept
    n_totals[basis]      = n_total
    n_kept_all[basis]    = n_kept
    p_succ = n_kept / n_total if n_total > 0 else 0
    n0 = kept.get('0', 0)
    n1 = kept.get('1', 0)
    p0 = n0 / n_kept if n_kept > 0 else float('nan')
    print(f"  {basis:>6}  {n_total:>12d}  {n_kept:>15d}  "
          f"{p_succ:>8.4f}  {p0:>10.4f}")

# =============================================================================
#  STEP 5 — Clopper–Pearson CI for p_succ
#           Use Z-basis run as the reference (same circuit, independent estimate)
# =============================================================================

def clopper_pearson(k, n, alpha=1 - CI_LEVEL):
    """Exact Clopper–Pearson CI for a binomial proportion."""
    lo = beta_dist.ppf(alpha / 2,     k,     n - k + 1) if k > 0 else 0.0
    hi = beta_dist.ppf(1 - alpha / 2, k + 1, n - k)     if k < n else 1.0
    return lo, hi

n_total_ref = n_totals['Z']
n_kept_ref  = n_kept_all['Z']
p_succ_hat  = n_kept_ref / n_total_ref
cp_lo, cp_hi = clopper_pearson(n_kept_ref, n_total_ref)

print(f"\n  p_succ (Z-basis run, Clopper–Pearson 95% CI):")
print(f"    p̂_succ = {p_succ_hat:.4f}  "
      f"[{cp_lo:.4f}, {cp_hi:.4f}]")
print(f"    (Expected ≈ 0.25 for single-qubit toy instance)")

# =============================================================================
#  STEP 6 — Reconstruct ρ_Y from post-selected tomography counts
#           Linear inversion on Pauli expectations + projection to valid DM
# =============================================================================

def pauli_expectation(counts_01, basis):
    """
    Compute <σ> from post-selected counts in a given Pauli basis.
    counts_01: dict {'0': n0, '1': n1}
    Returns expectation value in [-1, 1].
    """
    n0 = counts_01.get('0', 0)
    n1 = counts_01.get('1', 0)
    N  = n0 + n1
    if N == 0:
        return 0.0, N
    exp = (n0 - n1) / N
    return exp, N


def reconstruct_density_matrix(sx, sy, sz):
    """
    Linear inversion: ρ = (I + sx·X + sy·Y + sz·Z) / 2
    Then project to nearest valid density matrix (positive semidefinite,
    trace 1) using eigenvalue clipping.
    """
    I  = np.eye(2, dtype=complex)
    X  = np.array([[0, 1], [1, 0]], dtype=complex)
    Y  = np.array([[0, -1j], [1j, 0]], dtype=complex)
    Z  = np.array([[1, 0], [0, -1]], dtype=complex)
    rho = (I + sx * X + sy * Y + sz * Z) / 2.0

    # Project to valid DM (Smolin–Gambetta–Smith method)
    eigvals, eigvecs = np.linalg.eigh(rho)
    eigvals_clipped  = np.maximum(eigvals, 0)
    eigvals_clipped /= eigvals_clipped.sum()
    rho_valid = (eigvecs * eigvals_clipped) @ eigvecs.conj().T
    return rho_valid


sx, _ = pauli_expectation(postselected['X'], 'X')
sy, _ = pauli_expectation(postselected['Y'], 'Y')
sz, _ = pauli_expectation(postselected['Z'], 'Z')

rho_Y = reconstruct_density_matrix(sx, sy, sz)
print(f"\n  Reconstructed ρ_Y (Bloch vector):")
print(f"    ⟨X⟩ = {sx:+.4f}")
print(f"    ⟨Y⟩ = {sy:+.4f}")
print(f"    ⟨Z⟩ = {sz:+.4f}")
print(f"    Purity Tr(ρ²) = {np.trace(rho_Y @ rho_Y).real:.4f}")

# =============================================================================
#  STEP 7 — Target state ρ_M
# =============================================================================

qc_msg = QuantumCircuit(1)
qc_msg.u(THETA_MSG, VARPHI_MSG, 0.0, 0)
psi_M  = Statevector.from_instruction(qc_msg)
rho_M  = DensityMatrix(psi_M)

# Point estimate of fidelity
F_hat = state_fidelity(DensityMatrix(rho_Y), rho_M)
print(f"\n  F(ρ_Y, ρ_M) = {F_hat:.4f}  (point estimate)")

# =============================================================================
#  STEP 8 — Bootstrap CI for F(ρ_Y, ρ_M)
#
#  For each bootstrap resample:
#    - Resample post-selected counts in each basis
#    - Recompute Pauli expectations
#    - Reconstruct ρ_Y
#    - Compute fidelity
# =============================================================================

print(f"\n  Running {N_BOOTSTRAP} bootstrap resamples for F confidence interval ...")

def bootstrap_resample_counts(counts_dict, rng):
    """Multinomial resample of a counts dict."""
    keys   = list(counts_dict.keys())
    vals   = np.array([counts_dict[k] for k in keys], dtype=int)
    N      = vals.sum()
    probs  = vals / N
    new_counts = rng.multinomial(N, probs)
    return {k: int(v) for k, v in zip(keys, new_counts)}


bootstrap_F = np.zeros(N_BOOTSTRAP)

for i in range(N_BOOTSTRAP):
    bs_X = bootstrap_resample_counts(postselected['X'], RNG)
    bs_Y = bootstrap_resample_counts(postselected['Y'], RNG)
    bs_Z = bootstrap_resample_counts(postselected['Z'], RNG)

    sx_b, _ = pauli_expectation(bs_X, 'X')
    sy_b, _ = pauli_expectation(bs_Y, 'Y')
    sz_b, _ = pauli_expectation(bs_Z, 'Z')

    rho_b        = reconstruct_density_matrix(sx_b, sy_b, sz_b)
    bootstrap_F[i] = state_fidelity(DensityMatrix(rho_b), rho_M)

alpha    = 1 - CI_LEVEL
F_lo     = np.percentile(bootstrap_F, 100 * alpha / 2)
F_hi     = np.percentile(bootstrap_F, 100 * (1 - alpha / 2))
F_std    = np.std(bootstrap_F)
F_median = np.median(bootstrap_F)

print(f"  Bootstrap complete.")
print(f"    F_median = {F_median:.4f}")
print(f"    F_std    = {F_std:.4f}")
print(f"    95% CI   = [{F_lo:.4f}, {F_hi:.4f}]")

# =============================================================================
#  STEP 9 — Paper-ready summary
# =============================================================================

print("\n" + "=" * 65)
print("  PAPER-READY SUMMARY — Section IV A")
print("=" * 65)
print(f"""
  Backend                  : {BACKEND}
  Total shots per basis    : {SHOTS}
  Total shots (3 bases)    : {3 * SHOTS}

  Post-selection (Bell outcome crR=0, crG=0):
    Shots kept (Z basis)   : {n_kept_ref} / {n_total_ref}
    p̂_succ                : {p_succ_hat:.4f}
    95% CI (Clopper–Pearson): [{cp_lo:.4f}, {cp_hi:.4f}]
    Expected (ideal)       : 0.2500

  State tomography on output qubit Y:
    Pauli expectations     : ⟨X⟩={sx:+.4f}, ⟨Y⟩={sy:+.4f}, ⟨Z⟩={sz:+.4f}
    Purity Tr(ρ_Y²)       : {np.trace(rho_Y @ rho_Y).real:.4f}

  Output fidelity F(ρ_Y, ρ_M):
    Point estimate         : {F_hat:.4f}
    Bootstrap median       : {F_median:.4f}
    Bootstrap std          : {F_std:.4f}
    95% CI (bootstrap)     : [{F_lo:.4f}, {F_hi:.4f}]
    Ideal (noiseless)      : 1.0000
""")

# =============================================================================
#  STEP 10 — Save raw bitstrings for reproducibility
# =============================================================================

import json, datetime

save_payload = {
    "metadata": {
        "backend"        : BACKEND,
        "shots_per_basis": SHOTS,
        "seed_transpiler": SEED,
        "timestamp"      : datetime.datetime.utcnow().isoformat() + "Z",
        "theta_msg"      : THETA_MSG,
        "varphi_msg"     : VARPHI_MSG,
    },
    "raw_counts"       : raw_results,          # all bitstrings before post-selection
    "postselected"     : {b: dict(v) for b, v in postselected.items()},
    "p_succ"           : {
        "estimate"     : p_succ_hat,
        "ci_lo"        : cp_lo,
        "ci_hi"        : cp_hi,
        "method"       : "Clopper-Pearson 95%"
    },
    "bloch_vector"     : {"sx": sx, "sy": sy, "sz": sz},
    "rho_Y"            : rho_Y.tolist(),
    "fidelity"         : {
        "point_estimate": F_hat,
        "bootstrap_median": F_median,
        "bootstrap_std" : F_std,
        "ci_lo"         : F_lo,
        "ci_hi"         : F_hi,
        "method"        : f"Bootstrap {N_BOOTSTRAP} resamples, 95% percentile CI"
    },
    "bootstrap_F_samples": bootstrap_F.tolist()
}

outfile = "hardware_postselection_results.json"
with open(outfile, "w") as f:
    json.dump(save_payload, f, indent=2)

print(f"[✓] Raw results saved to '{outfile}'")
print(f"[✓] Pipeline complete.\n")

qiskit_runtime_service._discover_account:WARNING:2026-03-19 16:02:05,735: Loading account with the given token. A saved account will not be used.


  IBM Quantum Post-Selection Pipeline — Implementation A


qiskit_runtime_service.__init__:WARNING:2026-03-19 16:02:09,026: Instance was not set at service instantiation. Free and trial plan instances will be prioritized. Based on the following filters: (tags: None, region: us-east, eu-de), and available plans: (open), the available account instances are: CTCs. If you need a specific instance set it explicitly either by using a saved account with a saved default instance or passing it in directly to QiskitRuntimeService().
qiskit_runtime_service.backends:WARNING:2026-03-19 16:02:09,027: Using instance: CTCs, plan: open



[✓] Connected to ibm_torino
  Transpiling 3 tomography circuits (optimization_level=3) ...
    Basis Z: depth=82, 2q-gates=32
    Basis X: depth=87, 2q-gates=32
    Basis Y: depth=86, 2q-gates=32

  Submitting 3 circuits × 1000 shots to ibm_torino (Batch mode — open plan compatible) ...
    Submitting basis Z ... job_id = d6u5cgif84ks73ddsoqg
    Submitting basis X ... job_id = d6u5cgqf84ks73ddsor0
    Submitting basis Y ... job_id = d6u5cgov5rlc73f3iemg

    Waiting for basis Z ... done  (1000 total counts, 8 unique bitstrings)
      QPU time: 2.0000 s
    Waiting for basis X ... done  (1000 total counts, 8 unique bitstrings)
      QPU time: 2.0000 s
    Waiting for basis Y ... done  (1000 total counts, 8 unique bitstrings)
      QPU time: 2.0000 s

  Post-selection results:
   Basis   Total shots   Kept (Bell=00)    p_succ   p(0|kept)
  ------------------------------------------------------------
       Z          1000              241    0.2410      0.2573
       X          1000   

TypeError: Object of type complex is not JSON serializable

In [9]:
# =============================================================================
#  IBM Quantum Hardware Post-Selection Pipeline — Implementation A
#  Probabilistic Yoshida–Kitaev Decoder (Lloyd-type CTC Emulation)
#
#  Pipeline:
#    1. Build & transpile the circuit on ibm_torino
#    2. Run on IBM Quantum hardware, store raw bitstrings
#    3. Filter bitstrings on Bell outcome |Φ+⟩ = (crR=0, crG=0)
#    4. Estimate p_succ with Clopper–Pearson 95% CI
#    5. State tomography on output qubit Y (3 Pauli bases: X, Y, Z)
#    6. Reconstruct ρ_Y via linear inversion + MLE
#    7. Compute F(ρ_Y, ρ_M) with bootstrap 95% CI
#    8. Print paper-ready summary table
#
#  Backend  : ibm_torino  (133-qubit Heron r2)
#  Shots    : 9000 total  (~2250 post-selected expected)
#  Calibration date: retrieved at job submission time
# =============================================================================

# ── Imports ───────────────────────────────────────────────────────────────────
import numpy as np
import warnings
warnings.filterwarnings("ignore")
from scipy.stats import beta as beta_dist
from scipy.optimize import minimize
from collections import defaultdict

from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit.quantum_info import (Statevector, DensityMatrix,
                                  state_fidelity, Pauli, SparsePauliOp)
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler
from qiskit_ibm_runtime import Batch

# =============================================================================
#  CONFIG — edit only this block
# =============================================================================
IBM_TOKEN   = "QfkScNfX4bVJ5lXm0082x7F7J6vya3SF5LJZFNOYXqqO"
BACKEND     = "ibm_torino"
SHOTS       = 10000          # total shots
N_BOOTSTRAP = 2000          # bootstrap resamples for F confidence interval
CI_LEVEL    = 0.95          # confidence level
SEED        = 42
RNG         = np.random.default_rng(SEED)

# Message state parameters (same as your circuit)
THETA_MSG   = 2.5349076035276403
VARPHI_MSG  = 2.0022404587009195

# =============================================================================
#  STEP 1 — Build circuit components
# =============================================================================

def build_base_circuit(theta=THETA_MSG, varphi=VARPHI_MSG):
    """Core 7-qubit probabilistic decoder, no measurements."""
    C = QuantumRegister(1, 'C')
    E = QuantumRegister(1, 'E')
    R = QuantumRegister(1, 'R')
    G = QuantumRegister(1, 'G')
    M = QuantumRegister(1, 'M')
    A = QuantumRegister(1, 'A')
    Y = QuantumRegister(1, 'Y')

    qc = QuantumCircuit(C, E, R, G, M, A, Y)

    # Prepare message on M
    qc.u(theta, varphi, 0.0, M)
    # SWAP message into CTC register
    qc.swap(C, M)
    qc.barrier()
    # Bell pairs: (E,M), (R,G), (A,Y)
    qc.h(E);  qc.cx(E, M)
    qc.h(R);  qc.cx(R, G)
    qc.h(A);  qc.cx(A, Y)
    qc.barrier()
    # Scrambling unitary U on (C, E, R)
    qc.cz(C, R);  qc.cz(E, R);  qc.cz(C, E)
    qc.h(C);  qc.h(E);  qc.h(R)
    qc.cz(C, R);  qc.cz(C, E);  qc.cz(E, R)
    qc.barrier()
    # Probabilistic decoder: U† on (A, M, G)
    qc.cz(A, G);  qc.cz(M, A);  qc.cz(G, M)
    qc.h(A);  qc.h(M);  qc.h(G)
    qc.cz(A, G);  qc.cz(G, M);  qc.cz(M, A)
    qc.barrier()
    # Bell projection on (R, G)
    qc.cx(R, G)
    qc.h(R)
    qc.barrier()
    # SWAP output Y -> C, apply U† on C
    qc.swap(C, Y)
    qc.u(theta, varphi, 0.0, C).inverse()

    return qc, C, E, R, G, M, A, Y


def build_tomography_circuits():
    """
    Build 3 tomography circuits measuring register C in X, Y, Z bases.

    After SWAP(C,Y), register C holds ρ_M (the recovered message).
    We do NOT apply U†(C) here — that step maps ρ_M back to |0⟩ and is
    only present in the original circuit for the CTC loop closure, not for
    tomographic readout. Tomography measures C directly against ρ_M.

    Post-selection condition: crR = 0  AND  crG = 0  (Bell outcome |Φ+⟩)

    Classical register order (added last→first in bitstring, left→right):
        crTomo (last added) | crG | crR (first added)
    Bitstring format: "crTomo crG crR"   e.g. "0 0 0"

    Returns list of (basis_label, circuit) pairs.
    """
    circuits = []
    for basis in ['Z', 'X', 'Y']:
        C = QuantumRegister(1, 'C');  E = QuantumRegister(1, 'E')
        R = QuantumRegister(1, 'R');  G = QuantumRegister(1, 'G')
        M = QuantumRegister(1, 'M');  A = QuantumRegister(1, 'A')
        Yr = QuantumRegister(1, 'Y')
        crR    = ClassicalRegister(1, 'crR')
        crG    = ClassicalRegister(1, 'crG')
        crTomo = ClassicalRegister(1, 'crTomo')
        qc = QuantumCircuit(C, E, R, G, M, A, Yr, crR, crG, crTomo)

        # --- Prepare message on M ---
        qc.u(THETA_MSG, VARPHI_MSG, 0.0, M)
        qc.swap(C, M); qc.barrier()
        # --- Bell pairs ---
        qc.h(E);  qc.cx(E, M)
        qc.h(R);  qc.cx(R, G)
        qc.h(A);  qc.cx(A, Yr); qc.barrier()
        # --- Scrambling unitary ---
        qc.cz(C, R);  qc.cz(E, R);  qc.cz(C, E)
        qc.h(C);  qc.h(E);  qc.h(R)
        qc.cz(C, R);  qc.cz(C, E);  qc.cz(E, R); qc.barrier()
        # --- Probabilistic decoder U† ---
        qc.cz(A, G);  qc.cz(M, A);  qc.cz(G, M)
        qc.h(A);  qc.h(M);  qc.h(G)
        qc.cz(A, G);  qc.cz(G, M);  qc.cz(M, A); qc.barrier()
        # --- Bell projection on (R, G) ---
        qc.cx(R, G);  qc.h(R); qc.barrier()
        # --- SWAP: C now holds ρ_M (no U†!) ---
        qc.swap(C, Yr)
        # --- Post-selection measurements ---
        qc.measure(R, crR)
        qc.measure(G, crG)
        # --- Tomography rotation on C ---
        if basis == 'X':
            qc.h(C)
        elif basis == 'Y':
            qc.sdg(C)
            qc.h(C)
        # basis == 'Z': no rotation
        qc.measure(C, crTomo)
        circuits.append((basis, qc))

    return circuits


# =============================================================================
#  STEP 2 — Transpile all 3 tomography circuits
# =============================================================================

print("=" * 65)
print("  IBM Quantum Post-Selection Pipeline — Implementation A")
print("=" * 65)

service = QiskitRuntimeService(channel="ibm_quantum_platform", token=IBM_TOKEN)
backend = service.backend(BACKEND)
print(f"\n[✓] Connected to {BACKEND}")

tomo_circuits_raw = build_tomography_circuits()

print(f"  Transpiling 3 tomography circuits (optimization_level=3) ...")
tomo_circuits_transpiled = []
for basis, qc in tomo_circuits_raw:
    t_circ = transpile(qc, backend=backend,
                       optimization_level=3,
                       seed_transpiler=SEED)
    tomo_circuits_transpiled.append((basis, t_circ))
    print(f"    Basis {basis}: depth={t_circ.depth()}, "
          f"2q-gates={sum(1 for _,qa,_ in t_circ.data if len(qa)==2)}")

# =============================================================================
#  STEP 3 — Run on IBM Quantum hardware
#
#  NOTE: Sessions are not available on the IBM Quantum Open Plan.
#  We use Batch mode instead — supported on the open plan and recommended
#  for collections of independent circuits. All 3 circuits are submitted
#  together in one batch for efficiency.
# =============================================================================

def extract_counts(result):
    """
    Extract joint {bitstring: count} dict from a SamplerV2 PrimitiveResult.

    SamplerV2 stores each classical register as a separate BitArray under
    pub.data.<register_name>  — there is no single combined counts dict.
    We reconstruct joint bitstrings by zipping the per-shot arrays:

        bitstring = crTomo_bit | crG_bit | crR_bit   (leftmost = last added)

    This matches the register order the postselect() function expects.
    """
    from collections import Counter
    import numpy as np

    pub  = result[0]
    data = pub.data

    # Each register is a BitArray with shape (n_shots, n_bits_in_register)
    # All three registers are 1-bit wide so .array.flatten() gives (n_shots,)
    try:
        crT_arr = data.crTomo.array.flatten()   # shape (n_shots,)
        crG_arr = data.crG.array.flatten()
        crR_arr = data.crR.array.flatten()
    except AttributeError as e:
        # Fallback: print available attributes to help debug
        avail = [a for a in dir(data) if not a.startswith('_')
                 and hasattr(getattr(data, a), 'get_counts')]
        raise RuntimeError(
            f"Could not find expected registers (crTomo/crG/crR). "
            f"Available BitArray attributes: {avail}"
        ) from e

    # Build joint bitstring "crTomo crG crR" per shot and count
    joint = [f"{int(t)}{int(g)}{int(r)}"
             for t, g, r in zip(crT_arr, crG_arr, crR_arr)]
    return dict(Counter(joint))

print(f"\n  Submitting {len(tomo_circuits_transpiled)} circuits × {SHOTS} shots "
      f"to {BACKEND} (Batch mode — open plan compatible) ...")

raw_results = {}   # basis -> bitstring count dict
jobs        = {}   # basis -> job handle

with Batch(backend=backend) as batch:
    sampler = Sampler(mode=batch)
    for basis, t_circ in tomo_circuits_transpiled:
        print(f"    Submitting basis {basis} ...", end=" ", flush=True)
        jobs[basis] = sampler.run([t_circ], shots=SHOTS)
        print(f"job_id = {jobs[basis].job_id()}")

# Collect results — blocks until each job finishes
print("\n  Collecting results (waiting for IBM Quantum jobs) ...")
for basis in ['Z', 'X', 'Y']:
    print(f"    Waiting for basis {basis} ...", end=" ", flush=True)
    result = jobs[basis].result()
    counts = extract_counts(result)
    raw_results[basis] = counts
    total  = sum(counts.values())
    print(f"done  ({total} total counts, {len(counts)} unique bitstrings)")
    try:
        q_secs = jobs[basis].metrics()['usage']['quantum_seconds']
        print(f"      QPU time: {q_secs:.4f} s")
    except Exception:
        pass

# =============================================================================
#  STEP 4 — Store raw bitstrings and post-select on Bell outcome |Φ+⟩
#           Post-selection condition: crR == 0  AND  crG == 0
#
#  Bitstring format from SamplerV2:  "crTomo crG crR"
#  (Qiskit orders classical registers right-to-left in the bitstring:
#   rightmost = first register added = crR, then crG, then crTomo)
# =============================================================================

def parse_bitstring(bitstring):
    """
    Parse a Qiskit bitstring into (crTomo, crG, crR) bits.
    Handles both formats hardware and simulator may return:
      Spaced  (simulator): "0 1 0"  -> splits to ['0','1','0']
      Compact (hardware):  "010"    -> list gives ['0','1','0']
    Register order: crTomo (leftmost) | crG | crR (rightmost).
    """
    parts = bitstring.split(' ') if ' ' in bitstring else list(bitstring)
    if len(parts) != 3:
        raise ValueError(
            f"Unexpected bitstring '{bitstring}' — need 3 bits, got {len(parts)}."
        )
    return parts[0], parts[1], parts[2]   # crTomo, crG, crR


def postselect(counts_dict):
    """
    Filter counts keeping only shots where crR=0 AND crG=0  (Bell |Phi+>).
    Returns (postselected_tomo_counts, n_total, n_kept).
    The returned dict maps '0'/'1' (crTomo outcome) -> count.
    """
    kept    = defaultdict(int)
    n_total = 0
    n_kept  = 0
    for bitstring, count in counts_dict.items():
        n_total += count
        crTomo_bit, crG_bit, crR_bit = parse_bitstring(bitstring)
        if crR_bit == '0' and crG_bit == '0':
            kept[crTomo_bit] += count
            n_kept += count
    return dict(kept), n_total, n_kept


postselected = {}   # basis -> {'0': n0, '1': n1}
n_totals     = {}
n_kept_all   = {}

print("\n  Post-selection results:")
print(f"  {'Basis':>6}  {'Total shots':>12}  {'Kept (Bell=00)':>15}  "
      f"{'p_succ':>8}  {'p(0|kept)':>10}")
print("  " + "-" * 60)

for basis in ['Z', 'X', 'Y']:
    kept, n_total, n_kept = postselect(raw_results[basis])
    postselected[basis]  = kept
    n_totals[basis]      = n_total
    n_kept_all[basis]    = n_kept
    p_succ = n_kept / n_total if n_total > 0 else 0
    n0 = kept.get('0', 0)
    n1 = kept.get('1', 0)
    p0 = n0 / n_kept if n_kept > 0 else float('nan')
    print(f"  {basis:>6}  {n_total:>12d}  {n_kept:>15d}  "
          f"{p_succ:>8.4f}  {p0:>10.4f}")

# =============================================================================
#  STEP 5 — Clopper–Pearson CI for p_succ
#           Use Z-basis run as the reference (same circuit, independent estimate)
# =============================================================================

def clopper_pearson(k, n, alpha=1 - CI_LEVEL):
    """Exact Clopper–Pearson CI for a binomial proportion."""
    lo = beta_dist.ppf(alpha / 2,     k,     n - k + 1) if k > 0 else 0.0
    hi = beta_dist.ppf(1 - alpha / 2, k + 1, n - k)     if k < n else 1.0
    return lo, hi

n_total_ref = n_totals['Z']
n_kept_ref  = n_kept_all['Z']
p_succ_hat  = n_kept_ref / n_total_ref
cp_lo, cp_hi = clopper_pearson(n_kept_ref, n_total_ref)

print(f"\n  p_succ (Z-basis run, Clopper–Pearson 95% CI):")
print(f"    p̂_succ = {p_succ_hat:.4f}  "
      f"[{cp_lo:.4f}, {cp_hi:.4f}]")
print(f"    (Expected ≈ 0.25 for single-qubit toy instance)")

# =============================================================================
#  STEP 6 — Reconstruct ρ_Y from post-selected tomography counts
#           Linear inversion on Pauli expectations + projection to valid DM
# =============================================================================

def pauli_expectation(counts_01, basis):
    """
    Compute <σ> from post-selected counts in a given Pauli basis.
    counts_01: dict {'0': n0, '1': n1}
    Returns expectation value in [-1, 1].
    """
    n0 = counts_01.get('0', 0)
    n1 = counts_01.get('1', 0)
    N  = n0 + n1
    if N == 0:
        return 0.0, N
    exp = (n0 - n1) / N
    return exp, N


def reconstruct_density_matrix(sx, sy, sz):
    """
    Linear inversion: ρ = (I + sx·X + sy·Y + sz·Z) / 2
    Then project to nearest valid density matrix (positive semidefinite,
    trace 1) using eigenvalue clipping.
    """
    I  = np.eye(2, dtype=complex)
    X  = np.array([[0, 1], [1, 0]], dtype=complex)
    Y  = np.array([[0, -1j], [1j, 0]], dtype=complex)
    Z  = np.array([[1, 0], [0, -1]], dtype=complex)
    rho = (I + sx * X + sy * Y + sz * Z) / 2.0

    # Project to valid DM (Smolin–Gambetta–Smith method)
    eigvals, eigvecs = np.linalg.eigh(rho)
    eigvals_clipped  = np.maximum(eigvals, 0)
    eigvals_clipped /= eigvals_clipped.sum()
    rho_valid = (eigvecs * eigvals_clipped) @ eigvecs.conj().T
    return rho_valid


sx, _ = pauli_expectation(postselected['X'], 'X')
sy, _ = pauli_expectation(postselected['Y'], 'Y')
sz, _ = pauli_expectation(postselected['Z'], 'Z')

rho_Y = reconstruct_density_matrix(sx, sy, sz)
print(f"\n  Reconstructed ρ_Y (Bloch vector):")
print(f"    ⟨X⟩ = {sx:+.4f}")
print(f"    ⟨Y⟩ = {sy:+.4f}")
print(f"    ⟨Z⟩ = {sz:+.4f}")
print(f"    Purity Tr(ρ²) = {np.trace(rho_Y @ rho_Y).real:.4f}")

# =============================================================================
#  STEP 7 — Target state ρ_M
# =============================================================================

qc_msg = QuantumCircuit(1)
qc_msg.u(THETA_MSG, VARPHI_MSG, 0.0, 0)
psi_M  = Statevector.from_instruction(qc_msg)
rho_M  = DensityMatrix(psi_M)

# Point estimate of fidelity
F_hat = state_fidelity(DensityMatrix(rho_Y), rho_M)
print(f"\n  F(ρ_Y, ρ_M) = {F_hat:.4f}  (point estimate)")

# =============================================================================
#  STEP 8 — Bootstrap CI for F(ρ_Y, ρ_M)
#
#  For each bootstrap resample:
#    - Resample post-selected counts in each basis
#    - Recompute Pauli expectations
#    - Reconstruct ρ_Y
#    - Compute fidelity
# =============================================================================

print(f"\n  Running {N_BOOTSTRAP} bootstrap resamples for F confidence interval ...")

def bootstrap_resample_counts(counts_dict, rng):
    """Multinomial resample of a counts dict."""
    keys   = list(counts_dict.keys())
    vals   = np.array([counts_dict[k] for k in keys], dtype=int)
    N      = vals.sum()
    probs  = vals / N
    new_counts = rng.multinomial(N, probs)
    return {k: int(v) for k, v in zip(keys, new_counts)}


bootstrap_F = np.zeros(N_BOOTSTRAP)

for i in range(N_BOOTSTRAP):
    bs_X = bootstrap_resample_counts(postselected['X'], RNG)
    bs_Y = bootstrap_resample_counts(postselected['Y'], RNG)
    bs_Z = bootstrap_resample_counts(postselected['Z'], RNG)

    sx_b, _ = pauli_expectation(bs_X, 'X')
    sy_b, _ = pauli_expectation(bs_Y, 'Y')
    sz_b, _ = pauli_expectation(bs_Z, 'Z')

    rho_b        = reconstruct_density_matrix(sx_b, sy_b, sz_b)
    bootstrap_F[i] = state_fidelity(DensityMatrix(rho_b), rho_M)

alpha    = 1 - CI_LEVEL
F_lo     = np.percentile(bootstrap_F, 100 * alpha / 2)
F_hi     = np.percentile(bootstrap_F, 100 * (1 - alpha / 2))
F_std    = np.std(bootstrap_F)
F_median = np.median(bootstrap_F)

print(f"  Bootstrap complete.")
print(f"    F_median = {F_median:.4f}")
print(f"    F_std    = {F_std:.4f}")
print(f"    95% CI   = [{F_lo:.4f}, {F_hi:.4f}]")

# =============================================================================
#  STEP 9 — Paper-ready summary
# =============================================================================

print("\n" + "=" * 65)
print("  PAPER-READY SUMMARY — Section IV A")
print("=" * 65)
print(f"""
  Backend                  : {BACKEND}
  Total shots per basis    : {SHOTS}
  Total shots (3 bases)    : {3 * SHOTS}

  Post-selection (Bell outcome crR=0, crG=0):
    Shots kept (Z basis)   : {n_kept_ref} / {n_total_ref}
    p̂_succ                : {p_succ_hat:.4f}
    95% CI (Clopper–Pearson): [{cp_lo:.4f}, {cp_hi:.4f}]
    Expected (ideal)       : 0.2500

  State tomography on output qubit Y:
    Pauli expectations     : ⟨X⟩={sx:+.4f}, ⟨Y⟩={sy:+.4f}, ⟨Z⟩={sz:+.4f}
    Purity Tr(ρ_Y²)       : {np.trace(rho_Y @ rho_Y).real:.4f}

  Output fidelity F(ρ_Y, ρ_M):
    Point estimate         : {F_hat:.4f}
    Bootstrap median       : {F_median:.4f}
    Bootstrap std          : {F_std:.4f}
    95% CI (bootstrap)     : [{F_lo:.4f}, {F_hi:.4f}]
    Ideal (noiseless)      : 1.0000
""")

# =============================================================================
#  STEP 10 — Save raw bitstrings for reproducibility
# =============================================================================

import json, datetime
import numpy as np

class _NumpyEncoder(json.JSONEncoder):
    """Serialize numpy/complex types that stdlib json cannot handle."""
    def default(self, obj):
        if isinstance(obj, complex):
            return {"__complex__": True, "re": obj.real, "im": obj.imag}
        if isinstance(obj, np.integer):
            return int(obj)
        if isinstance(obj, np.floating):
            return float(obj)
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        return super().default(obj)

def _to_serializable(arr):
    """Convert a 2-D complex numpy array to [[re,im], ...] nested list."""
    return [[{"re": float(v.real), "im": float(v.imag)} for v in row]
            for row in np.array(arr)]

save_payload = {
    "metadata": {
        "backend"        : BACKEND,
        "shots_per_basis": SHOTS,
        "seed_transpiler": SEED,
        "timestamp"      : datetime.datetime.utcnow().isoformat() + "Z",
        "theta_msg"      : float(THETA_MSG),
        "varphi_msg"     : float(VARPHI_MSG),
    },
    "raw_counts"    : raw_results,
    "postselected"  : {b: dict(v) for b, v in postselected.items()},
    "p_succ"        : {
        "estimate"  : float(p_succ_hat),
        "ci_lo"     : float(cp_lo),
        "ci_hi"     : float(cp_hi),
        "method"    : "Clopper-Pearson 95%"
    },
    "bloch_vector"  : {
        "sx": float(sx), "sy": float(sy), "sz": float(sz)
    },
    "rho_Y"         : _to_serializable(rho_Y),
    "fidelity"      : {
        "point_estimate"  : float(F_hat),
        "bootstrap_median": float(F_median),
        "bootstrap_std"   : float(F_std),
        "ci_lo"           : float(F_lo),
        "ci_hi"           : float(F_hi),
        "method"          : f"Bootstrap {N_BOOTSTRAP} resamples, 95% percentile CI"
    },
    "bootstrap_F_samples": [float(v) for v in bootstrap_F],
}

outfile = "hardware_postselection_results.json"
with open(outfile, "w") as f:
    json.dump(save_payload, f, indent=2, cls=_NumpyEncoder)

print(f"[✓] Raw results saved to '{outfile}'")
print(f"[✓] Pipeline complete.\n")

qiskit_runtime_service._discover_account:WARNING:2026-03-19 16:09:35,650: Loading account with the given token. A saved account will not be used.


  IBM Quantum Post-Selection Pipeline — Implementation A


qiskit_runtime_service.__init__:WARNING:2026-03-19 16:09:38,463: Instance was not set at service instantiation. Free and trial plan instances will be prioritized. Based on the following filters: (tags: None, region: us-east, eu-de), and available plans: (open), the available account instances are: CTCs. If you need a specific instance set it explicitly either by using a saved account with a saved default instance or passing it in directly to QiskitRuntimeService().
qiskit_runtime_service.backends:WARNING:2026-03-19 16:09:38,465: Using instance: CTCs, plan: open



[✓] Connected to ibm_torino
  Transpiling 3 tomography circuits (optimization_level=3) ...
    Basis Z: depth=82, 2q-gates=32
    Basis X: depth=87, 2q-gates=32
    Basis Y: depth=86, 2q-gates=32

  Submitting 3 circuits × 10000 shots to ibm_torino (Batch mode — open plan compatible) ...
    Submitting basis Z ... job_id = d6u5g0qf84ks73ddss60
    Submitting basis X ... job_id = d6u5g10v5rlc73f3ii60
    Submitting basis Y ... job_id = d6u5g1atnsts73erpmpg

    Waiting for basis Z ... done  (10000 total counts, 8 unique bitstrings)
      QPU time: 4.0000 s
    Waiting for basis X ... done  (10000 total counts, 8 unique bitstrings)
      QPU time: 4.0000 s
    Waiting for basis Y ... done  (10000 total counts, 8 unique bitstrings)
      QPU time: 4.0000 s

  Post-selection results:
   Basis   Total shots   Kept (Bell=00)    p_succ   p(0|kept)
  ------------------------------------------------------------
       Z         10000             2430    0.2430      0.1988
       X         1000